# 02 — the book at one second, every venue

`gold.book_top20` is the state at the *end* of each second, replayed from every venue frame (Kraken's checksum-verified). Pick an instant and look at BTC across venues.

In [ ]:
from k2lake import connect, SCALE
con = connect()
AT = con.sql("SELECT max(second) - INTERVAL 30 MINUTE FROM lake.gold.book_top20").fetchone()[0]
print('at', AT)

In [ ]:
con.sql(f"""
SELECT exchange, canonical_symbol, second, depth, checksum_ok,
       bid_px_e8[1] / 100000000 AS bid, bid_qty_e8[1] / 100000000 AS bid_qty,
       ask_px_e8[1] / 100000000 AS ask, ask_qty_e8[1] / 100000000 AS ask_qty
FROM lake.gold.book_top20
WHERE canonical_symbol LIKE 'BTC/%' AND second = TIMESTAMP '{AT}'
ORDER BY exchange
""").show()

The whole ladder for one venue — 20 levels a side, quantities in base units. `unnest` turns the four arrays into rows.

In [ ]:
con.sql(f"""
WITH b AS (SELECT * FROM lake.gold.book_top20 WHERE exchange = 'kraken' AND canonical_symbol = 'BTC/USD' AND second = TIMESTAMP '{AT}')
SELECT i AS level, bid_qty_e8[i] / 100000000 AS bid_qty, bid_px_e8[i] / 100000000 AS bid, ask_px_e8[i] / 100000000 AS ask, ask_qty_e8[i] / 100000000 AS ask_qty
FROM b, range(1, 21) r(i)
WHERE i <= depth ORDER BY i
""").show(max_rows=20)

Spread and imbalance over the surrounding hour, from `gold.bbo_1s` (the same arithmetic ClickHouse's `gold.bbo_live` applies on read).

In [ ]:
import matplotlib.pyplot as plt
df = con.sql(f"""
SELECT exchange, second, spread_bps, imbalance FROM lake.gold.bbo_1s
WHERE canonical_symbol LIKE 'BTC/%' AND second BETWEEN TIMESTAMP '{AT}' - INTERVAL 30 MINUTE AND TIMESTAMP '{AT}' + INTERVAL 30 MINUTE
ORDER BY exchange, second
""").df()
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for ex, g in df.groupby('exchange'):
    ax[0].plot(g['second'], g['spread_bps'], label=ex, lw=0.7)
    ax[1].plot(g['second'], g['imbalance'], label=ex, lw=0.7)
ax[0].set_ylabel('spread (bps)'); ax[1].set_ylabel('L1 imbalance'); ax[0].legend(); plt.tight_layout()